In [32]:
import numpy as np
# import pandas as pd
import cv2
# import matplotlib.pyplot as plt
import os
# import torch
# import torchvision

In [23]:
# Define paths
train_images_folder_path = "../data/train/images" 
train_mask_folder_path = "../data/train/masks" 
test_images_folder_path = "../data/val/images" 
test_mask_folder_path = "../data/val/masks" 

In [24]:
# Get image and mask names
train_images_names_original = sorted([img for img in os.listdir(train_images_folder_path) if img.endswith('.png')])
train_mask_names_original = sorted([img for img in os.listdir(train_mask_folder_path) if img.endswith('.png')])
test_images_names = sorted([img for img in os.listdir(test_images_folder_path) if img.endswith('.png')])
test_mask_names = sorted([img for img in os.listdir(test_mask_folder_path) if img.endswith('.png')])

In [25]:
def one_hot_mask(mask):
    """Convert RGB mask to one-hot encoded tensor."""
    mask = np.array(mask)
    one_hot_map = []
    for color in colors:
        class_map = np.all(mask == color, axis=-1)
        one_hot_map.append(class_map)
    one_hot = np.stack(one_hot_map, axis=-1)
    return torch.from_numpy(one_hot.astype(np.int32)).permute(2, 0, 1)


In [26]:
def preprocess_image(img_path, height=96, width=256, normalize=True, crop=False):
    """Load and preprocess image."""
    img = Image.open(img_path).convert("RGB")

    if crop:
        img = img.crop((0, 0, 2048, 768))  
        
    img = img.resize((width, height))
    img = F.to_tensor(img)

    if normalize:  
        img = F.normalize(img, mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
    return img


In [27]:
def preprocess_mask(mask_path, height=96, width=256, one_hot=True, crop=False):
    """Load and preprocess mask."""
    mask = Image.open(mask_path).convert("RGB")

    if crop:
        mask = mask.crop((0, 0, 2048, 768))

    mask = mask.resize((width, height), resample=Image.NEAREST)

    if one_hot:
        mask = one_hot_mask(mask)
    else:
        mask = F.to_tensor(mask)

    return mask


In [28]:
def augment_image_mask(image, mask):
    """Apply same augmentation to both image and mask."""
    if torch.rand(1) < 0.5:
        image = F.hflip(image)
        mask = F.hflip(mask)

    angle = transforms.RandomRotation.get_params([-15, 15])
    image = F.rotate(image, angle)
    mask = F.rotate(mask, angle)

    jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
    image = jitter(image)

    return image, mask


In [29]:
def load_train(img_folder, mask_folder, img_name, mask_name):
    """Load train image and mask with augmentation pipeline."""
    img_path = os.path.join(img_folder, img_name)
    mask_path = os.path.join(mask_folder, mask_name)

    image = preprocess_image(img_path)
    mask = preprocess_mask(mask_path)

    image, mask = augment_image_mask(image, mask)
    return image, mask
